# YOLO 청구기호 라벨 검출기 **v4** — 909 부트스트랩 승격

**v3에서 바뀐 것:** 걷기 151프레임 채점에서 하이브리드만 못 읽은 10권(909~911 집중)의
휴리스틱 정답 프레임 31개를 train 강제 + ×3 복제 승격. 나머지 레시피(하단 40% 타이트 라벨,
증강, epochs)는 v3와 동일 — 변인 통제.

**데이터:** `yolo_labelset_v4.zip` (train 237 · val 31, 사진+동영상 206장 원본)
**목표:** 걷기 판독률 89% → 95% 돌파 (휴리스틱 동률 이상), 사진 회귀 없음(600_11 26·700_10 26 유지)
**런타임:** GPU(T4) · 약 40~50분

In [ ]:
# 1) 설치 + 데이터 업로드 (yolo_labelset_v4.zip)
!pip install -q ultralytics
from google.colab import files
up = files.upload()   # yolo_labelset_v4.zip
!unzip -oq yolo_labelset_v4.zip
!ls yolo_labelset_v4/images/train | wc -l; ls yolo_labelset_v4/images/val | wc -l

In [ ]:
# 2) 학습 — v3와 동일 하이퍼파라미터 (변인은 데이터셋뿐)
from ultralytics import YOLO
try:
    model = YOLO('yolo26n.pt')
except Exception as e:
    print('yolo26n 불가 → yolo11n 폴백:', e)
    model = YOLO('yolo11n.pt')
model.train(data='yolo_labelset_v4/data.yaml', epochs=120, imgsz=1280, batch=8,
            degrees=12, perspective=0.0008, shear=4, fliplr=0.0,
            hsv_v=0.5, hsv_s=0.4, mosaic=0.6, patience=30, name='call_label_v4')

In [ ]:
# 3) 검증 수치 + 시각 확인
m = YOLO('runs/detect/call_label_v4/weights/best.pt')
r = m.val(data='yolo_labelset_v4/data.yaml', imgsz=1280)
print('mAP50:', r.box.map50, 'mAP50-95:', r.box.map)
import glob
res = m.predict(glob.glob('yolo_labelset_v4/images/val/*.jpg')[0], imgsz=1280, conf=0.3, save=True)
print('예측 저장:', res[0].save_dir)

In [ ]:
# 4) ONNX 내보내기 + 다운로드
m.export(format='onnx', imgsz=1280, half=False, dynamic=False)
!zip -q -j call_label_yolo_v4.zip runs/detect/call_label_v4/weights/best.pt runs/detect/call_label_v4/weights/best.onnx
from google.colab import files
files.download('call_label_yolo_v4.zip')

## 5) 로컬 채점 (다운로드 후)

```bash
# call_label_yolo_v4.zip → libar-sample/call_label_yolo_v4/ 에 풀기
# ① 걷기 151프레임 재실행 (Colab GPU 배치 daelim_hybrid_v3_colab.ipynb의 YOLO 경로만 v4로)
# ② 채점: py -3.12 walk_grade.py --results hybrid_v4_results
#    통과선: 판독률 95%↑ (v3 89%, 휴리스틱 95%) — 특히 909-싱12ㅅ 등 10권 회수 여부
# ③ 사진 회귀 검사: 600_11·700_10 (v2 26·26 유지가 기준)
# 지면 유지: 지형별 챔피언 (사진=v2, 라이브=v3) — v4는 승자일 때만 교체
```

**이후 조각이 쌓이면:** 광수쌤 현장 스캔 조각은 `drive_crops_to_pairs.py`로 rec 학습쌍이 됨
(검출기가 아니라 **인식기 v5** 재료 — 조각엔 전체 사진이 없어 YOLO 학습엔 못 씀).
검출기 재료가 더 필요하면 광수쌤께 **서가 사진**(000~500번대 스팟)을 요청.